In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"]="API-Token"

In [ ]:
CNN_model=r"C:\Users\Neeraj\Desktop\chest_front\my_model_2.keras"
image_size=(256,256)
classes=["Normal","Pneumonia"]

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import numpy as np
from PIL import Image
import tensorflow as tf
cnn_model=tf.keras.models.load_model(CNN_model)
print(cnn_model.summary())

In [ ]:
def preprocess_xray(image_path: str) -> np.ndarray:
    img=Image.open(image_path).convert("RGB")
    img=img.resize(image_size)
    arr=np.array(img, dtype=np.float32)/255.0
    return np.expand_dims(arr,axis=0)

In [ ]:
def classify_xray(image_path: str) -> tuple[str, float, dict]:
    img_tensor=preprocess_xray(image_path)
    raw=cnn_model.predict(img_tensor, verbose=0)[0][0]
    pred_idx=int(raw > 0.5)
    predicted_class=classes[pred_idx]
    confidence=float(raw) if pred_idx == 1 else float(1 - raw)
    all_probs={"Normal": float(1 - raw), "Pneumonia": float(raw)}

    return predicted_class, confidence, all_probs

In [ ]:

from langchain_community.document_loaders import PyPDFLoader

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
loader=PyPDFLoader(r"/content/chest_docs.pdf")

In [ ]:
docs=loader.load()

In [ ]:
print(len(docs))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120,separators=["\n\n", "\n", ". ", " ", ""],)

In [ ]:
chunks=splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace
from langchain_community.vectorstores import FAISS
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)
vector_store=FAISS.from_documents(chunks,embedding_model)

In [ ]:
vector_store.save_local("faiss_chest_index")

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
retriever.invoke("Precautions of Pneumonia")

In [ ]:
!pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
import os
os.environ["OPENAI_API_KEY"]="API_Token"
model=ChatOpenAI(model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",temperature=0.7)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
prompt_template = """\
You are a cautious, evidence-based medical assistant AI.
You provide guidance on chest conditions detected from X-rays.

STRICT RULES:
- Base your response ONLY on the CONTEXT provided.
- If information is missing, say "Not enough information in the knowledge base."
- NEVER give a definitive prescription. Only suggest possible approaches.
- Always recommend consulting a qualified doctor.
- Pay special attention to safety considerations for children and pregnant women.
- If the condition is "Normal", reassure but still recommend a doctor visit.

━━━━━━━━━━━━━━━━━━━━━━
PATIENT PROFILE:
- Detected Condition : {condition}
- CNN Confidence     : {confidence}%
- Patient Age        : {age} years old
- Pregnant           : {pregnant}
━━━━━━━━━━━━━━━━━━━━━━

KNOWLEDGE BASE CONTEXT:
{context}

━━━━━━━━━━━━━━━━━━━━━━
Please provide a structured response with these sections:

1. 📋 Condition Summary
   Brief explanation of the detected condition.

2. ⚠️  Severity Assessment
   Likely severity based on the context. Note confidence level.

3. 💊 Suggested Treatment Approaches
   Based ONLY on the context. Include possible medicines with dosage ranges if mentioned.

4. 👶🤰 Special Considerations
   Specific guidance for the patient's age group (child) or pregnancy status.
   List any medicines/treatments to AVOID for this patient.

5. 🏥 When to Seek Immediate Help
   Warning signs that require emergency care.

6. 🔒 Safety Disclaimer
   Remind that this is AI guidance only and a doctor must be consulted.

ANSWER:
"""

In [ ]:
prompt=PromptTemplate(
    input_variables=["condition", "confidence", "context", "age", "pregnant"],
    template=prompt_template,
)

In [ ]:
question=f"""
Pneumonia detected from chest X-ray.
Patient age: {age}
Pregnant: {pregnancy_status}
What is the recommended treatment?
"""

In [ ]:
def format_docs(docs) -> str:
    return "\n\n---\n\n".join(
        f"[Source:{d.metadata.get('source','unknown')},Page {d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs
    )

In [ ]:
def run_pipeline(
    image_path: str,
    patient_age: int,
    is_pregnant: bool,
) -> dict:
  condition, confidence, all_probs=classify_xray(image_path)
  print(f"\n🔬 CNN Result: {condition} ({confidence*100:.1f}% confidence)")
  print(f"All probs: { {k: f'{v*100:.1f}%' for k,v in all_probs.items()} }")
  query=(
      f"{condition} treatment guidelines "
      +("children pediatric" if patient_age < 18 else "adult")
      +(" pregnancy pregnant women" if is_pregnant else "")
      +f" medications precautions management"
    )
  retrieved_docs=retriever.invoke(query)
  context=format_docs(retrieved_docs)
  print(f"\n📚 Retrieved {len(retrieved_docs)} relevant chunks from knowledge base.")
  chain=prompt | model | StrOutputParser()
  response=chain.invoke({
      "condition":condition,
      "confidence":f"{confidence*100:.1f}",
      "age":patient_age,
      "pregnant":"Yes" if is_pregnant else "No",
      "context":context,
  })
  return {
      "condition":condition,
      "confidence":confidence,
      "all_probs":all_probs,
      "response":response,
  }


In [ ]:
test_image_path="/content/right-lung-pneumonia.jpg"   
patient_age=int(input("Enter patient age: "))
preg_input=input("Is the patient pregnant? (yes/no): ").strip().lower()
is_pregnant=preg_input in ("yes", "y", "true")
result = run_pipeline(test_image_path,patient_age,is_pregnant)
print("\n"+"═"*60)
print("MEDICAL REPORT")
print("═"*60)
print(result["response"])
print("═"*60)